In [4]:
!pip install -q transformers==4.51.3 peft trl bitsandbytes datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 17.5 MB/s eta 0:00:00


In [2]:
import os
os.makedirs("data", exist_ok=True)

In [5]:
import os
os.makedirs("/root/.cache/huggingface/accelerate", exist_ok=True)

with open("/root/.cache/huggingface/accelerate/default_config.yaml", "w") as f:
    f.write("""compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
downcast_bf16: 'no'
mixed_precision: fp16
num_machines: 1
num_processes: 1
use_cpu: false
""")

print("Accelerate config written.")

Accelerate config written.


In [10]:
%%writefile train.py
from __future__ import annotations
import argparse, inspect, os
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

# Force fp16 at the process level — must happen before accelerate initializes
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"

DEFAULT_DATA_DIR = Path(__file__).resolve().parent / "data"
DEFAULT_OUTPUT_DIR = Path(__file__).resolve().parent / "outputs" / "lora"
DEFAULT_TARGET_MODULES = ("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")

def _format_chat(example, tokenizer) -> str:
    return tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", default="Qwen/Qwen2.5-1.5B-Instruct")
    parser.add_argument("--train-file", type=Path, default=DEFAULT_DATA_DIR / "train.jsonl")
    parser.add_argument("--eval-file",  type=Path, default=DEFAULT_DATA_DIR / "eval.jsonl")
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT_DIR)
    parser.add_argument("--max-seq-length", type=int, default=2048)
    parser.add_argument("--epochs",        type=float, default=3.0)
    parser.add_argument("--batch-size",    type=int, default=1)
    parser.add_argument("--grad-accum",    type=int, default=8)
    parser.add_argument("--learning-rate", type=float, default=2e-4)
    parser.add_argument("--lora-r",        type=int, default=8)
    parser.add_argument("--lora-alpha",    type=int, default=16)
    parser.add_argument("--lora-dropout",  type=float, default=0.05)
    parser.add_argument("--target-modules", default=",".join(DEFAULT_TARGET_MODULES))
    args = parser.parse_args()

    # ── Tokenizer ──────────────────────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(args.model, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # ── Quantization (4-bit NF4, fp16 compute) ────────────────────────────────
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,   # ← fp16, never bfloat16
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    # ── Model ──────────────────────────────────────────────────────────────────
    model = AutoModelForCausalLM.from_pretrained(
        args.model,
        quantization_config=bnb_config,
        device_map={"": 0},                     # pin to GPU 0
        torch_dtype=torch.float16,
    )
    model.config.torch_dtype = torch.float16
    model.config.use_cache = False

    # Cast any non-quantized layers (norms, embeddings) that default to bfloat16
    for name, param in model.named_parameters():
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)

    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    # ── Dataset ────────────────────────────────────────────────────────────────
    dataset = load_dataset("json", data_files={
        "train": str(args.train_file),
        "eval":  str(args.eval_file),
    })
    dataset = dataset.map(
        lambda row: {"text": _format_chat(row, tokenizer)},
        remove_columns=dataset["train"].column_names,
    )

    # ── LoRA ───────────────────────────────────────────────────────────────────
    peft_config = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[m.strip() for m in args.target_modules.split(",") if m.strip()],
    )

    # ── Training args (dynamic to survive TRL version changes) ────────────────
    sft_kwargs = dict(
        output_dir=str(args.output_dir),
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.learning_rate,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=100,
        save_total_limit=2,
        bf16=False,                             # ← never bfloat16
        fp16=True,                              # ← always fp16 on T4
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        report_to="none",
        dataset_text_field="text",
        packing=False,
    )

    sig = inspect.signature(SFTConfig.__init__).parameters
    sft_kwargs["max_seq_length" if "max_seq_length" in sig else "max_length"] = args.max_seq_length
    training_args = SFTConfig(**sft_kwargs)

    # ── Trainer ────────────────────────────────────────────────────────────────
    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["eval"],
        peft_config=peft_config,
    )
    tsig = inspect.signature(SFTTrainer.__init__).parameters
    trainer_kwargs["processing_class" if "processing_class" in tsig else "tokenizer"] = tokenizer

    trainer = SFTTrainer(**trainer_kwargs)
    trainer.train()
    trainer.save_model(str(args.output_dir))
    tokenizer.save_pretrained(str(args.output_dir))

if __name__ == "__main__":
    main()

Overwriting train.py


In [11]:
!python train.py \
  --model "Qwen/Qwen2.5-1.5B-Instruct"

2026-05-12 21:57:07.095525: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
average_tokens_across_devices is set to True but it is invalid when world size is1. Turn it to False automatically.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
{'loss': 1.8421, 'grad_norm': 0.5927836298942566, 'learning_rate': 0.00015384615384615385, 'num_tokens': 59676.0, 'mean_token_accuracy': 0.6387942679226398, 'epoch': 0.74}
{'loss': 1.02, 'grad_norm':

In [2]:
!pip install -q huggingface_hub
from huggingface_hub import login
login()

In [13]:
ls /content/outputs/lora

adapter_config.json        checkpoint-39/           tokenizer_config.json
adapter_model.safetensors  merges.txt               tokenizer.json
added_tokens.json          README.md                training_args.bin
checkpoint-13/             special_tokens_map.json  vocab.json


In [15]:
!pip install -q torchao --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.8 MB/s eta 0:00:00


In [3]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "outputs/lora"  # adjust path
HF_REPO = "alinamoca25/hikelogic-qwen2.5-1.5b"

# Load base + merge LoRA weights in
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, OUTPUT_DIR)
merged = model.merge_and_unload()  # bakes LoRA into base weights

merged.push_to_hub(HF_REPO)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
tokenizer.push_to_hub(HF_REPO)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...6fgoo30/model.safetensors:   0%|          | 21.6kB / 3.09GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp6vy985i2/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

CommitInfo(commit_url='https://huggingface.co/alinamoca25/hikelogic-qwen2.5-1.5b/commit/52c2a6d63cee87ad3a9117a1295e440c3678aeb7', commit_message='Upload tokenizer', commit_description='', oid='52c2a6d63cee87ad3a9117a1295e440c3678aeb7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/alinamoca25/hikelogic-qwen2.5-1.5b', endpoint='https://huggingface.co', repo_type='model', repo_id='alinamoca25/hikelogic-qwen2.5-1.5b'), pr_revision=None, pr_num=None)

In [4]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
import torch

login()

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER = "alinamoca25/hikelogic-qwen2.5-1.5b"
MERGED_REPO = "alinamoca25/hikelogic-qwen2.5-1.5b-merged"

# Load base in fp16 (no quantization this time)
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER)

# Load and merge
model = PeftModel.from_pretrained(base, ADAPTER)
merged = model.merge_and_unload()

# Push merged model
merged.push_to_hub(MERGED_REPO)
tokenizer.push_to_hub(MERGED_REPO)
print("Done!")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpcwc4pei0/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Done!
